In [1]:
!git clone https://github.com/Ulyssesllc/ulysses-gpt.git
%cd /kaggle/working/ulysses-gpt
!pip install -e ".[dev]"


Cloning into 'ulysses-gpt'...
remote: Enumerating objects: 189, done.
remote: Counting objects: 100% (189/189), done.
remote: Compressing objects: 100% (115/115), done.
remote: Total 189 (delta 101), reused 144 (delta 56), pack-reused 0 (from 0)
Receiving objects: 100% (189/189), 65.29 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (101/101), done.
/kaggle/working/ulysses-gpt
Obtaining file:///kaggle/working/ulysses-gpt
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.5/15.5 MB 60.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 77.2 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 31.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.7/531.7 kB 14.2 MB/s eta 0:00:0000:01
  Building editable 

In [2]:
!ls -R

.:
check_code.sh  LICENSE	       README.md	 run_project.sh  tests
Dockerfile     pyproject.toml  requirements.txt  src		 train.py

./src:
mini_gpt  ulysses_gpt.egg-info

./src/mini_gpt:
dataset.py   hf_export.py  metrics.py  post_train.py
generate.py  __init__.py   model.py    trainer.py

./src/ulysses_gpt.egg-info:
dependency_links.txt  PKG-INFO	    SOURCES.txt
entry_points.txt      requires.txt  top_level.txt

./tests:
test_model.py  test_post_train.py  test_trainer.py


In [3]:
!bash check_code.sh

🔍 [1/3] Running Ruff Linter & Formatter...
All checks passed!
13 files already formatted
✓ Ruff checks passed!

🔬 [2/3] Running Mypy Static Type Checker...
Success: no issues found in 8 source files
✓ Mypy type check passed!

🧪 [3/3] Running Pytest Unit Tests...
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /kaggle/working/ulysses-gpt
configfile: pyproject.toml
plugins: cov-7.1.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 15 items                                                             

tests/test_model.py::test_gpt_config_defaults PASSED                     [  6%]
tests/test_model.py::test_causal_self_attention_shape PASSED             [ 13%]
tests/test_model.py::test_mlp_shape PASSED                               [ 20%]
tests/test_model.py::test_transformer_block_shape PASSED                 [ 26%]
tests/test_model.p

In [ ]:
!python train.py --task pretrain --hf-dataset Salesforce/wikitext --hf-config wikitext-2-raw-v1 --hf-split train --validation-split validation --text-column text --block-size 256 --batch-size 2 --grad-accum-steps 4 --epochs 1 --max-steps 2000 --max-tokens 20000000 --max-val-tokens 2000000 --n-layer 4 --n-head 4 --n-embd 256 --learning-rate 0.0003 --checkpoint-dir checkpoints

Running on device: CUDA
Total parameters: 16,090,880
Trainable parameters: 16,090,880
Loading Hugging Face dataset 'Salesforce/wikitext' (split='train')...
Loaded 36,718 training rows.
Using text column 'text'
Prepared shuffled training split.
Loading validation split 'validation'...
Loaded 3,760 validation rows.
Using dataset validation split 'validation'
 Encoding train (limit: 20,000,000 tokens)...
Tokenizing train: 93batch [00:09, 10.15batch/s]
 Encoded 2,370,618 tokens from train.
 Encoding validation (limit: 2,000,000 tokens)...
Tokenizing validation: 10batch [00:00, 12.94batch/s]
 Encoded 245,125 tokens from validation.
Epoch 1: 100%|████| 8000/8000 [02:37<00:00, 50.91it/s, loss=6.4967, lr=3.00e-05]


In [6]:
%%bash
set -euo pipefail

# Pretraining: WikiText-2
export PRETRAIN_HF_DATASET="Salesforce/wikitext"
export PRETRAIN_HF_CONFIG="wikitext-2-raw-v1"
export PRETRAIN_HF_SPLIT="train"
export VALIDATION_SPLIT="validation"
export TEXT_COLUMN="text"
export MAX_TOKENS=20000000
export MAX_VAL_TOKENS=2000000

# Cấu hình mô hình và mức dùng GPU phù hợp để bắt đầu trên T4
export CHECKPOINT_DIR="/kaggle/working/ulysses-gpt/checkpoints"
export BLOCK_SIZE=256
export BATCH_SIZE=2
export GRAD_ACCUM_STEPS=4
export N_LAYER=4
export N_HEAD=4
export N_EMBD=256
export EPOCHS=1
export MAX_STEPS=2000
export LEARNING_RATE=0.0003
export SEED=42
export PROMPT="Hello"

# SFT: Dolly-15k
export SFT_HF_DATASET="databricks/databricks-dolly-15k"
export SFT_HF_SPLIT="train"
export SFT_PROMPT_COLUMN="instruction"
export SFT_CONTEXT_COLUMN="context"
export RESPONSE_COLUMN="response"
export SFT_EPOCHS=1
export SFT_MAX_SAMPLES=2000
export SFT_LR=0.0001

# DPO
export DPO_HF_DATASET="CultriX/preference-dataset"
export DPO_HF_SPLIT="train"
export PROMPT_COLUMN="prompt"
export CHOSEN_COLUMN="chosen"
export REJECTED_COLUMN="rejected"
export DPO_EPOCHS=1
export DPO_MAX_SAMPLES=1000
export DPO_LR=0.000005

bash run_project.sh

Stage 1/3: pretraining
Running on device: CUDA
Total parameters: 16,090,880
Trainable parameters: 16,090,880
Loading Hugging Face dataset 'Salesforce/wikitext' (split='train')...
Loaded 36,718 training rows.
Using text column 'text'
Prepared shuffled training split.
Loading validation split 'validation'...
Loaded 3,760 validation rows.
Using dataset validation split 'validation'
 Encoding train (limit: 20,000,000 tokens)...
Tokenizing train: 93batch [00:08, 10.63batch/s]
 Encoded 2,370,618 tokens from train.
 Encoding validation (limit: 2,000,000 tokens)...
Tokenizing validation: 10batch [00:00, 12.24batch/s]
 Encoded 245,125 tokens from validation.
Epoch 1: 100%|██████████| 8000/8000 [02:27<00:00, 54.40it/s, loss=6.4967, lr=3.00e-05]
Epoch 1: train=6.4296, val=6.1781, ppl=482.10, next-token-acc=19.03%
Sample: Hello .
Inference: TTFT=0.731s, throughput=2.53 tokens/s
Stage 2/3: supervised fine-tuning (SFT)
Running on device: CUDA
Total parameters: 16,090,880
Trainable parameters: 16,090

In [ ]:
!python train.py 